# Hather: train the student (Qwen3.5-0.8B)

The student learns to answer like the teacher did in Run B: the same system prompt, the same
four-line input (`[SOURCE]`, `[SENDER]`, `[SIGNALS]`, `[TEXT]`), and the teacher's JSON answer.
Loss is on the answer only. LoRA rank 16 on all linear layers, 16-bit weights (no 4-bit: Unsloth
advises against QLoRA for Qwen3.5), 2 epochs, max 1,024 tokens.

**Before running:** Accelerator **GPU T4 x2** (one GPU is used), Internet **on**, input: your private
training dataset with `train_balanced.jsonl` and `val.jsonl`. *Save Version → Save & Run All*.

**Slow on purpose:** the T4 has no bfloat16 and Qwen3.5 gives NaNs in float16, so Unsloth trains it in
float32 on a T4. The progress lines print an ETA. Checkpoints are saved every 200 steps: if the run
stops, attach this version's output and run again; training continues from the last checkpoint.

In [ ]:
# 0. Settings + dead switch (before anything slow)
import os, glob, json, time, shutil, subprocess, re, random
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # Unsloth trains on one GPU; set before torch loads
NOTEBOOK_VERSION = "2026-10-27a"
print("notebook version:", NOTEBOOK_VERSION)

MODEL_NAME = "unsloth/Qwen3.5-0.8B"
MAX_LEN    = 1024
EPOCHS     = 2
LR         = 2e-4
BATCH, GRAD_ACCUM = 4, 2                           # effective batch 8
SAVE_STEPS = 200
QUICK_TEST = 120                                   # val messages answered by the trained model at the end
PUSH_TO_HF = False                                 # True: upload the GGUF to a PRIVATE Hugging Face repo
HF_REPO    = "hather-0.8b"                         #   (needs a Kaggle secret named HF_TOKEN with write access)
WORK = "/kaggle/working"
CKPT_DIR, LORA_DIR, GGUF_DIR = f"{WORK}/checkpoints", f"{WORK}/hather-0.8b-lora", f"{WORK}/hather-0.8b-gguf"

problems = []
def find(name):
    f = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not f: problems.append(f"{name} not found under /kaggle/input (Add Input -> your training dataset)")
    return f[0] if f else None
TRAIN_FILE, VAL_FILE = find("train_balanced.jsonl"), find("val.jsonl")
gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()
if not gpus: problems.append("no GPU: set Accelerator to GPU T4 x2")
if shutil.disk_usage(WORK).free < 6e9: problems.append("less than 6 GB free in /kaggle/working")
if problems:
    print("STOPPING:\n - " + "\n - ".join(problems)); raise SystemExit(1)

def load(p): return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
TRAIN, VAL = load(TRAIN_FILE), load(VAL_FILE)
bad = [r.get("id") for r in TRAIN + VAL if [m["role"] for m in r.get("messages", [])] != ["system", "user", "assistant"]]
if bad:
    print(f"STOPPING: {len(bad)} examples are not system/user/assistant chats, e.g. {bad[:3]}"); raise SystemExit(1)
systems = {r["messages"][0]["content"] for r in TRAIN + VAL}
print(f"OK: {len(TRAIN)} train, {len(VAL)} val, {len(systems)} system prompt(s), GPUs: {gpus}")

In [ ]:
# 1. Install Unsloth (the install cell from Unsloth's own Kaggle notebook for Qwen3.5)
import importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [ ]:
# 2. Load the model in 16-bit and add LoRA adapters (settings from Unsloth's Qwen3.5 fine-tuning guide)
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_LEN,
    load_in_4bit = False,          # QLoRA is not recommended for Qwen3.5
    load_in_16bit = True,
    full_finetuning = False,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    max_seq_length = MAX_LEN,
)
tok = getattr(tokenizer, "tokenizer", tokenizer)    # Qwen3.5 is a vision-language model: use its text tokenizer
print("bf16 supported:", is_bfloat16_supported(), "| GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 3. Data: prompt = exactly what the app will send (system + input, chat template, thinking OFF);
#    completion = the teacher's JSON + end-of-turn. Loss is computed on the completion only.
def render(messages, generation):
    kw = dict(tokenize = False, add_generation_prompt = generation, enable_thinking = False)
    try:
        return tok.apply_chat_template(messages, **kw)
    except Exception:              # some templates want content as a list of parts
        msgs = [{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]} for m in messages]
        return tok.apply_chat_template(msgs, **kw)

def to_pair(ex):
    prompt = render(ex["messages"][:2], True)
    answer = ex["messages"][2]["content"]
    full = render(ex["messages"], False)
    if full.startswith(prompt) and answer in full[len(prompt):]:
        completion = full[len(prompt):]                 # the template's own ending for an assistant turn
        how = "template"
    else:
        completion = answer + "<|im_end|>\n"
        how = "manual"
    return {"prompt": prompt, "completion": completion, "id": ex["id"], "how": how}

pairs_train = [to_pair(ex) for ex in TRAIN]
pairs_val = [to_pair(ex) for ex in VAL]
print("completion built from:", dict(__import__("collections").Counter(p["how"] for p in pairs_train)))
print("\n--- one example, as the model sees it ---\n" + pairs_train[0]["prompt"][-400:] + "[[COMPLETION]]" + pairs_train[0]["completion"])

def n_tokens(p): return len(tok(p["prompt"] + p["completion"], add_special_tokens = False)["input_ids"])
lens = [n_tokens(p) for p in pairs_train]
too_long = sum(l > MAX_LEN for l in lens)
print(f"\ntokens per example: median {sorted(lens)[len(lens)//2]}, max {max(lens)}; over {MAX_LEN}: {too_long}")
keep = lambda ps: [{"prompt": p["prompt"], "completion": p["completion"]} for p in ps if n_tokens(p) <= MAX_LEN]

from datasets import Dataset
train_ds = Dataset.from_list(keep(pairs_train))
val_rows = keep(pairs_val)
random.Random(0).shuffle(val_rows)
eval_ds = Dataset.from_list(val_rows[:300])           # a fixed slice keeps evaluation quick in float32
print(f"train {len(train_ds)}, eval {len(eval_ds)} (of {len(val_rows)} val)")

In [ ]:
# 4. Trainer, with an ETA line every 25 steps and a checkpoint every SAVE_STEPS steps
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback

class ETA(TrainerCallback):
    def on_train_begin(self, args, state, control, **kw):
        self.t0, self.s0 = time.time(), state.global_step
    def on_log(self, args, state, control, logs=None, **kw):
        done = state.global_step - self.s0
        if done and state.global_step % 25 == 0:
            per = (time.time() - self.t0) / done
            left = (state.max_steps - state.global_step) * per
            print(f"step {state.global_step}/{state.max_steps}  {per:.1f}s/step  ETA {left/3600:.1f} h  "
                  f"loss {logs.get('loss', logs.get('eval_loss', '-'))}", flush=True)

steps_per_epoch = -(-len(train_ds) // (BATCH * GRAD_ACCUM))
trainer = SFTTrainer(
    model = model,
    tokenizer = tok,
    train_dataset = train_ds,
    eval_dataset = eval_ds,
    callbacks = [ETA()],
    args = SFTConfig(
        per_device_train_batch_size = BATCH,
        per_device_eval_batch_size = BATCH,
        gradient_accumulation_steps = GRAD_ACCUM,
        num_train_epochs = EPOCHS,
        learning_rate = LR,
        warmup_steps = 20,
        lr_scheduler_type = "linear",
        optim = "adamw_8bit",
        weight_decay = 0.001,
        bf16 = is_bfloat16_supported(),
        fp16 = not is_bfloat16_supported(),     # Unsloth switches Qwen3.5 to float32 on a T4 (no NaNs)
        logging_steps = 5,
        eval_strategy = "steps",
        eval_steps = steps_per_epoch // 2,      # twice per epoch
        save_strategy = "steps",
        save_steps = SAVE_STEPS,
        save_total_limit = 2,
        output_dir = CKPT_DIR,
        max_length = MAX_LEN,
        completion_only_loss = True,            # loss on the teacher's answer only
        dataset_num_proc = 1,
        report_to = "none",
        seed = 3407,
    ),
)
print(f"{steps_per_epoch} steps per epoch, {steps_per_epoch * EPOCHS} in total")

In [ ]:
# 5. Check the loss mask on one example: only the teacher's JSON may count
batch = trainer.data_collator([trainer.train_dataset[0]])
ids, labels = batch["input_ids"][0], batch["labels"][0]
learned = tok.decode(ids[labels != -100])
ignored = tok.decode(ids[labels == -100])
print("LEARNED:", learned[:300]); print("IGNORED (end):", ignored[-200:])
if '"verdict"' not in learned or "[TEXT]" in learned or "fraud analyst" in learned:
    raise SystemExit("STOPPING: the loss mask is wrong (the prompt is being learned or the answer is not). "
                     "Do not train; send the lines above.")
print("loss mask OK")

In [ ]:
# 6. Train (continues from the last checkpoint if an earlier version's output is attached)
prev = sorted(glob.glob("/kaggle/input/**/checkpoints/checkpoint-*", recursive=True),
              key=lambda p: int(p.rsplit("-", 1)[-1]))
resume = None
if prev:
    resume = f"{CKPT_DIR}/{os.path.basename(prev[-1])}"
    shutil.copytree(prev[-1], resume, dirs_exist_ok=True)
    print("continuing from", prev[-1])
gpu = torch.cuda.get_device_properties(0)
print(f"GPU {gpu.name}, {gpu.total_memory/1e9:.1f} GB")
stats = trainer.train(resume_from_checkpoint = resume)
print(f"done in {stats.metrics['train_runtime']/3600:.2f} h, peak memory {torch.cuda.max_memory_reserved()/1e9:.1f} GB")
model.save_pretrained(LORA_DIR); tok.save_pretrained(LORA_DIR)
print("LoRA adapters saved to", LORA_DIR)

In [ ]:
# 7. Quick test: the trained student answers QUICK_TEST val messages it never trained on
#    (val messages are teacher-checked answers; the real score comes later from your test set)
from collections import Counter
FastLanguageModel.for_inference(model)
ARABIC = re.compile(r"[ء-ي]")
def ar(t):
    l = [c for c in t if c.isalpha()]; return sum(bool(ARABIC.match(c)) for c in l) / max(1, len(l))

sample = random.Random(1).sample(VAL, min(QUICK_TEST, len(VAL)))
res, t0 = Counter(), time.time()
for i, ex in enumerate(sample):
    prompt = render(ex["messages"][:2], True)
    enc = tok(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**enc, max_new_tokens=300, do_sample=False, use_cache=True)
    text = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    want = json.loads(ex["messages"][2]["content"])["verdict"]
    try:
        ans = json.loads(text[text.index("{"): text.rindex("}") + 1])
        ok = ans.get("verdict") == want
        res[(ex["language"], want, "right" if ok else "wrong")] += 1
        msg_ar = ar(ex["messages"][1]["content"].split("[TEXT]", 1)[-1])
        if (msg_ar >= 0.8 and ar(ans.get("explanation", "")) < 0.5) or (msg_ar < 0.2 and ar(ans.get("explanation", "")) >= 0.5):
            res["wrong language"] += 1
        if len(ans.get("explanation", "").split()) > 40: res["explanation over 40 words"] += 1
    except Exception:
        res["not valid JSON"] += 1
    if i < 3: print(f"\n[{want}] {text[:300]}")
print(f"\n{len(sample)} answers in {(time.time()-t0)/60:.1f} min")
for lang in ("ar", "en"):
    for lab in ("scam", "legit"):
        r, w = res[(lang, lab, "right")], res[(lang, lab, "wrong")]
        if r + w: print(f"{lang} {lab:5}: {r}/{r+w} right ({r/(r+w):.0%})")
print("not valid JSON:", res["not valid JSON"], "| wrong language:", res["wrong language"],
      "| explanation over 40 words:", res["explanation over 40 words"])

In [ ]:
# 8. Export for the phone: GGUF q4_k_m (about 0.5 GB). If the export fails, a merged 16-bit copy is saved instead.
try:
    model.save_pretrained_gguf(GGUF_DIR, tok, quantization_method = "q4_k_m")
except Exception as e:
    print("GGUF export failed:", type(e).__name__, str(e)[:300])
    print("saving a merged 16-bit model instead (convert it to GGUF later)")
    model.save_pretrained_merged(f"{WORK}/hather-0.8b-merged16", tok, save_method = "merged_16bit")
ggufs = glob.glob(f"{WORK}/**/*.gguf", recursive=True)
for g in ggufs: print(f"{g}  {os.path.getsize(g)/1e6:.0f} MB")

if PUSH_TO_HF and ggufs:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi
    token = UserSecretsClient().get_secret("HF_TOKEN")
    api = HfApi(token = token)
    repo = f"{api.whoami()['name']}/{HF_REPO}"
    api.create_repo(repo, private = True, exist_ok = True)        # private: the model saw real phone messages
    for g in ggufs: api.upload_file(path_or_fileobj = g, path_in_repo = os.path.basename(g), repo_id = repo)
    print("uploaded to the private repo", repo)

**When it finishes**, download from the Output tab into `hather/models/` (keep it out of git):
- the `.gguf` file (the phone model) and the `hather-0.8b-lora/` folder (small; lets you continue or re-export);
- copy the log lines of cell 7 (the quick test) into chat.

Then the real score: run the student on your labelled test set (`eval/`), which it has never seen.